In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score,recall_score,f1_score
import tensorflow_hub as hub

c:\Users\Sakib Malik\Desktop\ml-projects\.venv\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


## Load Dataset

In [2]:
df=pd.read_csv('cleaned_data.csv')
df.head()

,label,Text
0,0,meeting ã¼ rite ill go home lor ã¼ dun feel li...
1,0,ill always even spirit ill get bb soon trying ...
2,0,sorry took long omw
3,0,thk 50 shd ok said plus minus 10 ã¼ leave line...
4,0,dunno juz askin cos got card got 20 4 salon ca...


## Split Dataset

In [3]:

X=df['Text']
y=df['label']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

X_train_np=X_train.to_numpy()
X_test_np=X_test.to_numpy()
y_train_np=y_train.to_numpy()
y_test_np=y_test.to_numpy()

## Compute stasticstic from vectorization

In [10]:
df["Text"] = df["Text"].fillna("").astype(str)

X = df["Text"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

avg_words_len = round(
    sum(len(text.split()) for text in df["Text"]) / len(df["Text"])
)

total_words_length=len(set(" ".join(df['Text']).split()))


print(f"Data Loaded. Training samples: {len(X_train_np)}")
print(f"Average words per message: {avg_words_len}")
print(f"Approximate vocabulary size: {total_words_length}")

Data Loaded. Training samples: 1195
Average words per message: 12
Approximate vocabulary size: 4717


## Helper function for traning and evaluation

In [11]:
def compile_and_fit(model, epochs=5):
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        X_train_np,
        y_train_np,
        epochs=epochs,
        validation_data=(X_test_np, y_test_np)
    )
    return history

def get_metrics(model, X, y):
    y_preds = np.round(model.predict(X))
    return {
        'accuracy': accuracy_score(y, y_preds),
        'precision': precision_score(y, y_preds),
        'recall': recall_score(y, y_preds),
        'f1-score': f1_score(y, y_preds)
    }

## Create the TextVectorization layer

In [12]:
from tensorflow.keras.layers import TextVectorization

text_vec = TextVectorization(
    max_tokens=total_words_length,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=avg_words_len,
)
text_vec.adapt(X_train_np)

## Model 1 – Dense embedding model (build and train)

In [13]:
input_layer = layers.Input(shape=(1,), dtype=tf.string)
x = text_vec(input_layer)
x = layers.Embedding(input_dim=total_words_length, output_dim=128)(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation='relu')(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)

model_1 = keras.Model(input_layer, output_layer, name="Dense_Model")
history_1 = compile_and_fit(model_1)
history_1

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.8678 - loss: 0.5684 - val_accuracy: 0.8863 - val_loss: 0.4780
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9155 - loss: 0.3706 - val_accuracy: 0.8963 - val_loss: 0.3222
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9481 - loss: 0.2117 - val_accuracy: 0.9130 - val_loss: 0.2380
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9782 - loss: 0.1192 - val_accuracy: 0.9264 - val_loss: 0.2000
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9841 - loss: 0.0824 - val_accuracy: 0.9365 - val_loss: 0.1816


## Model 2 – Bidirectional LSTM model (build and train)

In [14]:
input_layer=layers.Input(shape=(1,),dtype=tf.string)
x=text_vec(input_layer)
x=layers.Embedding(input_dim=total_words_length,output_dim=128)(x)
x=layers.Bidirectional(layers.LSTM(64,return_sequences=True))(x)
x=layers.Bidirectional(layers.LSTM(64))(x)
x=layers.Flatten()(x)
x=layers.Dropout(0.1)(x)
x=layers.Dense(32,activation="relu")(x)
output_layer=layers.Dense(1,activation="sigmoid")(x)


model_2=keras.Model(input_layer,output_layer,name="BiLstm_model")
history_2=compile_and_fit(model_2)
history_2


Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 15s 88ms/step - accuracy: 0.8335 - loss: 0.4165 - val_accuracy: 0.9064 - val_loss: 0.2766
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9582 - loss: 0.1279 - val_accuracy: 0.9264 - val_loss: 0.2242
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.9900 - loss: 0.0393 - val_accuracy: 0.9331 - val_loss: 0.2616
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.9975 - loss: 0.0142 - val_accuracy: 0.9298 - val_loss: 0.2978
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/step - accuracy: 0.9992 - loss: 0.0030 - val_accuracy: 0.9197 - val_loss: 0.3640
